In [1]:
!apt-get update
!apt-get install -y yosys iverilog
!yosys --version
!iverilog -V

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,311 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,970 kB]
Get:14 http:

In [2]:
!wget -O nangate45.lib https://raw.githubusercontent.com/The-OpenROAD-Project/OpenROAD-flow-scripts/master/flow/platforms/nangate45/lib/NangateOpenCellLibrary_typical.lib
!ls -lh nangate45.lib

--2026-04-18 11:30:07--  https://raw.githubusercontent.com/The-OpenROAD-Project/OpenROAD-flow-scripts/master/flow/platforms/nangate45/lib/NangateOpenCellLibrary_typical.lib
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6692032 (6.4M) [text/plain]
Saving to: ‘nangate45.lib’

nangate45.lib       100%[===================>]   6.38M  --.-KB/s    in 0.08s   

2026-04-18 11:30:07 (81.7 MB/s) - ‘nangate45.lib’ saved [6692032/6692032]

-rw-r--r-- 1 root root 6.4M Apr 18 11:30 nangate45.lib


In [3]:
!git clone https://github.com/FCHXWH823/Verilog-Adders.git
!find Verilog-Adders -iname "KSA8.v" -o -iname "KSA8.v"

Cloning into 'Verilog-Adders'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 147 (delta 31), reused 53 (delta 21), pack-reused 80 (from 1)
Receiving objects: 100% (147/147), 952.25 KiB | 16.14 MiB/s, done.
Resolving deltas: 100% (65/65), done.
Verilog-Adders/Kogge-Stone Adder/KSA8.v


In [4]:
!cp Verilog-Adders/*/RCA8.v .
!cp Verilog-Adders/*/KSA8.v .
!ls

ksa8_generated.v  nangate45.lib  sample_data		 Verilog-Adders
KSA8.v		  RCA8.v	 testbench_final_ksa8.v


In [5]:
%%writefile constraints.sdc
create_clock -name clk -period 2.0
set_input_delay 0.2 -clock clk [all_inputs]
set_output_delay 0.2 -clock clk [all_outputs]

Writing constraints.sdc


In [6]:
%%writefile synth_adder.ys
read_verilog $env(ADDER_FILE)
hierarchy -check -top $env(TOP_MODULE)

flatten

proc; opt; fsm; opt; memory; opt
techmap; opt
dfflibmap -liberty nangate45.lib
abc -liberty nangate45.lib -constr constraints.sdc
clean
stat -liberty nangate45.lib

Writing synth_adder.ys


In [7]:
%%writefile run_yosys.py
import subprocess, re, json, sys

def synthesize(verilog_file, top_module, lib_file='nangate45.lib'):
    script = f"""
read_verilog {verilog_file}
hierarchy -check -top {top_module}

flatten

proc; opt; fsm; opt; memory; opt
techmap; opt
dfflibmap -liberty {lib_file}
abc -liberty {lib_file} -constr constraints.sdc
clean
stat -liberty {lib_file}
"""
    with open('temp_synth.ys', 'w') as f:
        f.write(script)

    result = subprocess.run(
        ['yosys', '-s', 'temp_synth.ys'],
        capture_output=True,
        text=True
    )

    log = result.stdout + result.stderr

    if result.returncode != 0:
        raise RuntimeError(log)

    return parse_stats(log)

def parse_stats(log):
    ppa = {}

    m = re.search(r'Chip area for.*?:\s+([\d.]+)', log)
    ppa['area_um2'] = float(m.group(1)) if m else None

    m = re.search(r'Number of cells:\s+(\d+)', log)
    ppa['cell_count'] = int(m.group(1)) if m else None

    m = re.search(r'Longest topological path.*?\((\d+) levels?\)', log, re.S)
    ppa['logic_levels'] = int(m.group(1)) if m else None

    return ppa

if __name__ == '__main__':
    ppa = synthesize(sys.argv[1], sys.argv[2])
    print(json.dumps(ppa, indent=2))

Writing run_yosys.py


In [8]:
!python run_yosys.py KSA8.v KSA8

{
  "area_um2": 70.224,
  "cell_count": 41,
  "logic_levels": null
}


In [9]:
!python run_yosys.py KSA8.v KSA8

{
  "area_um2": 70.224,
  "cell_count": 41,
  "logic_levels": null
}


In [10]:
%%writefile optimize_ksa.py
import json, os, sys
from openai import OpenAI
from run_yosys import synthesize

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def get_mode_prompt(mode):
    common = """You are an expert digital circuit designer.

Generate ONLY valid synthesizable Verilog for an 8-bit Kogge-Stone adder.

Strict requirements:
1. Keep the top-level module name exactly KSA8.
2. Preserve a Kogge-Stone / parallel-prefix style architecture.
3. Do NOT convert the design into RCA, CLA, or any other adder family.
4. Keep all internal propagate/generate signals explicitly declared.
5. Do not use undeclared signals.
6. Do not use out-of-bounds bus indexing.
7. Return ONLY valid Verilog code.
8. No markdown fences, no explanation.
"""

    if mode == "area":
        return common + """
Optimization goal:
- Minimize cell count and area.
- Delay target is relaxed (about 14 logic levels).
"""
    elif mode == "delay":
        return common + """
Optimization goal:
- Prefer lower delay if possible.
- Target about 6 logic levels, even if area increases somewhat.
"""
    elif mode == "balanced":
        return common + """
Optimization goal:
- Balanced PPA.
- Target about 10 logic levels if feasible while also minimizing cell count.
"""
    else:
        raise ValueError("mode must be one of: area, delay, balanced")

def clean_verilog(verilog_text):
    verilog_text = verilog_text.replace("```verilog", "").replace("```", "").strip()
    start = verilog_text.find("module")
    end = verilog_text.rfind("endmodule")
    if start != -1 and end != -1:
        verilog_text = verilog_text[start:end + len("endmodule")]
    return verilog_text.strip()

def llm_generate(system_prompt, history):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_prompt}] + history
    )
    return clean_verilog(response.choices[0].message.content)

def build_feedback_prompt(mode, iteration, ppa, best_ppa, top_module):
    if mode == "area":
        goal = "Reduce cell count and area. Delay can be relaxed up to about 14 logic levels."
    elif mode == "delay":
        goal = "Reduce delay aggressively toward about 6 logic levels, even if area increases somewhat."
    else:
        goal = "Find a balanced design: about 10 logic levels if possible, while also reducing cell count."

    return f"""Iteration {iteration} synthesis results:
- Cell count: {ppa['cell_count']}
- Area (um^2): {ppa['area_um2']}
- Logic levels: {ppa['logic_levels']}

Best so far:
- Cell count: {best_ppa['cell_count']}
- Area (um^2): {best_ppa['area_um2']}
- Logic levels: {best_ppa['logic_levels']}

Goal for next iteration:
{goal}

Important requirements:
- Keep the top-level module name exactly {top_module}
- Preserve KSA8 / parallel-prefix structure
- Do not output RCA8
- Declare all internal signals explicitly
- Avoid undefined signals and out-of-bounds indexing
- Return only valid Verilog code
"""

def is_better(mode, ppa, best_ppa):
    if ppa["cell_count"] is None or ppa["area_um2"] is None:
        return False

    if mode == "area":
        return (
            ppa["cell_count"] < best_ppa["cell_count"] or
            (ppa["cell_count"] == best_ppa["cell_count"] and ppa["area_um2"] < best_ppa["area_um2"])
        )

    if mode == "delay":
        if ppa["logic_levels"] is not None and best_ppa["logic_levels"] is not None:
            return (
                ppa["logic_levels"] < best_ppa["logic_levels"] or
                (ppa["logic_levels"] == best_ppa["logic_levels"] and ppa["area_um2"] < best_ppa["area_um2"])
            )
        return ppa["area_um2"] < best_ppa["area_um2"]

    if mode == "balanced":
        if ppa["logic_levels"] is not None and best_ppa["logic_levels"] is not None:
            return (
                ppa["logic_levels"] < best_ppa["logic_levels"] or
                (ppa["logic_levels"] == best_ppa["logic_levels"] and ppa["cell_count"] < best_ppa["cell_count"])
            )
        return (
            ppa["cell_count"] < best_ppa["cell_count"] or
            (ppa["cell_count"] == best_ppa["cell_count"] and ppa["area_um2"] < best_ppa["area_um2"])
        )

    return False

def run_loop(baseline_file, top_module, mode, max_iter=10):
    system_prompt = get_mode_prompt(mode)

    with open(baseline_file, "r") as f:
        baseline_verilog = f.read()

    baseline_ppa = synthesize(baseline_file, top_module)

    history = [{
        "role": "user",
        "content": f"""Here is my current 8-bit adder design:

{baseline_verilog}

Optimize this design in {mode} mode.
Preserve functional correctness and keep the top module name {top_module}.
Preserve a Kogge-Stone / parallel-prefix architecture.
Do not convert it into RCA8.
Your first proposal may be identical."""
    }]

    best_ppa = baseline_ppa
    best_code = baseline_verilog
    results = [{"iteration": 0, "ppa": baseline_ppa, "file": baseline_file}]

    for i in range(1, max_iter + 1):
        print(f"\n=== Iteration {i} ===")
        verilog = llm_generate(system_prompt, history)

        fname = f"ksa_{mode}_candidate_{i}.v"
        with open(fname, "w") as f:
            f.write(verilog)

        try:
            ppa = synthesize(fname, top_module)
        except Exception as e:
            print("Synthesis failed:", e)
            history.append({"role": "assistant", "content": verilog})
            history.append({
                "role": "user",
                "content": f"""Synthesis failed.

Fix all syntax, hierarchy, undeclared signal, and indexing issues.
Important:
- keep module name exactly {top_module}
- preserve KSA8 / parallel-prefix architecture
- do not output RCA8
- declare all internal signals explicitly
- avoid out-of-bounds indexing
Return valid Verilog only."""
            })
            continue

        print(f"Cells: {ppa['cell_count']}, Area: {ppa['area_um2']}, Levels: {ppa['logic_levels']}")
        results.append({"iteration": i, "ppa": ppa, "file": fname})

        if is_better(mode, ppa, best_ppa):
            best_ppa = ppa
            best_code = verilog
            print("*** New best! ***")

        history.append({"role": "assistant", "content": verilog})
        history.append({"role": "user", "content": build_feedback_prompt(mode, i, ppa, best_ppa, top_module)})

    best_name = f"best_ksa_{mode}.v"
    log_name = f"optimization_log_ksa_{mode}.json"

    with open(best_name, "w") as f:
        f.write(best_code)

    with open(log_name, "w") as f:
        json.dump({"mode": mode, "best_ppa": best_ppa, "iterations": results}, f, indent=2)

    print(f"\nBest ({mode}):", best_ppa)

if __name__ == "__main__":
    baseline_file = sys.argv[1]
    top_module = sys.argv[2]
    mode = sys.argv[3]
    run_loop(baseline_file, top_module, mode, max_iter=10)

Writing optimize_ksa.py


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [34]:
!python optimize_ksa.py ksa8_generated.v KSA8 area


=== Iteration 1 ===
Cells: 20, Area: 25.004, Levels: None
*** New best! ***

=== Iteration 2 ===
Cells: 16, Area: 22.078, Levels: None
*** New best! ***

=== Iteration 3 ===
Cells: 12, Area: 18.088, Levels: None
*** New best! ***

=== Iteration 4 ===
Cells: 19, Area: 26.334, Levels: None

=== Iteration 5 ===
Cells: 19, Area: 27.93, Levels: None

=== Iteration 6 ===
Cells: 16, Area: 23.408, Levels: None

=== Iteration 7 ===
Cells: 20, Area: 28.196, Levels: None

=== Iteration 8 ===
Cells: 20, Area: 28.196, Levels: None

=== Iteration 9 ===
Cells: 19, Area: 27.132, Levels: None

=== Iteration 10 ===
Cells: 20, Area: 28.196, Levels: None

Best (area): {'area_um2': 18.088, 'cell_count': 12, 'logic_levels': None}


In [19]:
!python optimize_ksa.py ksa8_generated.v KSA8 delay
!python optimize_ksa.py ksa8_generated.v KSA8 balanced


=== Iteration 1 ===
Cells: 23, Area: 36.176, Levels: None

=== Iteration 2 ===
Cells: 23, Area: 36.176, Levels: None

=== Iteration 3 ===
Cells: 23, Area: 36.176, Levels: None

=== Iteration 4 ===
Cells: 23, Area: 36.176, Levels: None

=== Iteration 5 ===
Cells: 23, Area: 36.176, Levels: None

=== Iteration 6 ===
Cells: 25, Area: 38.836, Levels: None

=== Iteration 7 ===
Cells: 26, Area: 35.644, Levels: None
*** New best! ***

=== Iteration 8 ===
Cells: 32, Area: 43.624, Levels: None

=== Iteration 9 ===
Cells: 28, Area: 41.762, Levels: None

=== Iteration 10 ===
Cells: 29, Area: 40.166, Levels: None

Best (delay): {'area_um2': 35.644, 'cell_count': 26, 'logic_levels': None}

=== Iteration 1 ===
Cells: 23, Area: 36.176, Levels: None

=== Iteration 2 ===
Cells: 29, Area: 45.486, Levels: None

=== Iteration 3 ===
Cells: 23, Area: 36.176, Levels: None

=== Iteration 4 ===
Cells: 34, Area: 48.678, Levels: None

=== Iteration 5 ===
Cells: 27, Area: 53.732, Levels: None

=== Iteration 6 ===

In [35]:
%%writefile plot_ppa.py
import json
import matplotlib.pyplot as plt

with open('optimization_log_ksa_balanced.json') as f:
    log = json.load(f)

iters = [r['iteration'] for r in log['iterations']]
cells = [r['ppa']['cell_count'] for r in log['iterations']]
areas = [r['ppa']['area_um2'] for r in log['iterations']]

plt.figure(figsize=(8,4))
plt.plot(iters, cells, marker='o')
plt.xlabel('Iteration')
plt.ylabel('Cell count')
plt.title('Cell Count Trajectory')
plt.grid(True)
plt.savefig('cell_trajectory.pdf')
plt.show()

plt.figure(figsize=(8,4))
plt.plot(iters, areas, marker='s')
plt.xlabel('Iteration')
plt.ylabel('Area (um^2)')
plt.title('Area Trajectory')
plt.grid(True)
plt.savefig('area_trajectory.pdf')
plt.show()

Overwriting plot_ppa.py


In [36]:
!python plot_ppa.py

Figure(800x400)
Figure(800x400)


In [27]:
!iverilog -o opt_sim best_ksa_balanced.v testbench_final_ksa8.v
!vvp opt_sim

VCD info: dumpfile testbench.vcd opened for output.
Test 0: a=0x00, b=0x00, sum=0x00, cout=0
✓ Test Passed: Expected sum=0x00, cout=0
Test 1: a=0x01, b=0x01, sum=0x02, cout=0
✓ Test Passed: Expected sum=0x02, cout=0
Test 2: a=0x0f, b=0x01, sum=0x0c, cout=0
✗ Test Failed: Expected sum=0x10, cout=0, Got sum=0x0c, cout=0
Test 3: a=0xff, b=0x01, sum=0xfc, cout=0
✗ Test Failed: Expected sum=0x00, cout=1, Got sum=0xfc, cout=0
Test 4: a=0xff, b=0x01, sum=0xfc, cout=0
✗ Test Failed: Expected sum=0x00, cout=1, Got sum=0xfc, cout=0
Test 5: a=0xaa, b=0x55, sum=0xff, cout=0
✓ Test Passed: Expected sum=0xff, cout=0
Test 6: a=0xcc, b=0x33, sum=0xff, cout=0
✓ Test Passed: Expected sum=0xff, cout=0
Test 7: a=0x81, b=0x7e, sum=0xff, cout=0
✓ Test Passed: Expected sum=0xff, cout=0
Test 8: a=0xf0, b=0x0f, sum=0xff, cout=0
✓ Test Passed: Expected sum=0xff, cout=0
Test 9: a=0x55, b=0xaa, sum=0xff, cout=0
✓ Test Passed: Expected sum=0xff, cout=0
Total Tests Run:          10
Total Passed Tests:           7
T

In [32]:
%%writefile equiv_check.ys
read_verilog -sv ksa8_generated.v
rename KSA8 gold

read_verilog -sv -ignore_redef best_ksa_balanced.v
rename KSA8 gate

equiv_make gold gate equiv
prep -top equiv
equiv_simple
equiv_status

Overwriting equiv_check.ys


In [33]:
!yosys -s equiv_check.ys


 /----------------------------------------------------------------------------\
 |                                                                            |
 |  yosys -- Yosys Open SYnthesis Suite                                       |
 |                                                                            |
 |  Copyright (C) 2012 - 2019  Clifford Wolf <clifford@clifford.at>           |
 |                                                                            |
 |  Permission to use, copy, modify, and/or distribute this software for any  |
 |  purpose with or without fee is hereby granted, provided that the above    |
 |  copyright notice and this permission notice appear in all copies.         |
 |                                                                            |
 |  THE SOFTWARE IS PROVIDED "AS IS" AND THE AUTHOR DISCLAIMS ALL WARRANTIES  |
 |  WITH REGARD TO THIS SOFTWARE INCLUDING ALL IMPLIED WARRANTIES OF          |
 |  MERCHANTABILITY AND FITNESS. IN NO 